In [ ]:
!pip install transformers torch peft datasets -

In [ ]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer,AutoModelForMultipleChoice,TrainingArguments,Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [ ]:
CHOICES    = ["A", "B", "C", "D", "E"]
LABEL2IDX  = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128

In [ ]:
train = pd.read_csv("train.csv")
train[CHOICES]    = train[CHOICES].fillna("").astype(str)
train["prompt"]   = train["prompt"].fillna("").astype(str)
train["answer"]   = train["answer"].str.strip().str.upper()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
train["label"] = train["answer"].map(LABEL2IDX)

label_150 = train.iloc[150]["label"]
answer_150 = train.iloc[150]["answer"]

print(f"\nQ1 — Row 150 answer letter : {answer_150}")
print(f"Q1 Answer — Encoded label  : {int(label_150)}")

In [ ]:
row0      = train.iloc[0]
prompt_0  = str(row0["prompt"])
option_b0 = str(row0["B"])

formatted = prompt_0 + " [SEP] " + option_b0
char_len  = len(formatted)

print(f"\nQ2 — Prompt: {prompt_0[:60]}...")
print(f"Q2 — Option B: {option_b0}")
print(f"Q2 — Formatted: {formatted[:80]}...")
print(f"Q2 Answer — Character length: {char_len}")

In [ ]:
def format_options(row):
    """Create 5 formatted (prompt [SEP] option) strings for one row."""
    prompt = str(row["prompt"])
    return [prompt + " [SEP] " + str(row[c]) for c in CHOICES]

In [ ]:
options_row0 = format_options(row0)

encoding_row0 = tokenizer(
    options_row0,
    padding       = "max_length",
    truncation    = True,
    max_length    = MAX_LEN,
    return_tensors= "pt",
)

input_ids_row0 = encoding_row0["input_ids"].unsqueeze(0)
second_dim     = input_ids_row0.shape[1]

print(f"\nQ3 — input_ids shape: {input_ids_row0.shape}")
print(f"Q3 Answer — Second dimension (num choices): {second_dim}")

In [ ]:
batch16 = train.iloc[:16]

all_input_ids = []
for _, row in batch16.iterrows():
    opts = format_options(row)
    enc  = tokenizer(
        opts,
        padding       = "max_length",
        truncation    = True,
        max_length    = MAX_LEN,
        return_tensors= "pt",
    )
    all_input_ids.append(enc["input_ids"].unsqueeze(0))   # [1, 5, 128]

# Stack into [16, 5, 128]
batch_input_ids = torch.cat(all_input_ids, dim=0)
total_positions = batch_input_ids.numel()   # 16 × 5 × 128

print(f"\nQ4 — Batch input_ids shape : {batch_input_ids.shape}")
print(f"Q4 Answer — Total token positions: {total_positions}")

In [ ]:

mc_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
mc_model.eval()

# Prepare input for row 0 → [1, 5, 128]
options_row0 = format_options(row0)
enc_row0 = tokenizer(
    options_row0,
    padding       = "max_length",
    truncation    = True,
    max_length    = MAX_LEN,
    return_tensors= "pt",
)
input_ids_mc      = enc_row0["input_ids"].unsqueeze(0)       # [1, 5, 128]
attention_mask_mc = enc_row0["attention_mask"].unsqueeze(0)  # [1, 5, 128]

with torch.no_grad():
    outputs_q5 = mc_model(
        input_ids      = input_ids_mc,
        attention_mask = attention_mask_mc,
    )

logits_shape  = outputs_q5.logits.shape   # [1, 5]
num_logits    = logits_shape[-1]

print(f"\nQ5 — Logits tensor shape : {logits_shape}")
print(f"Q5 Answer — Number of logits per question: {num_logits}")

In [ ]:
label_row0 = torch.tensor([int(train.iloc[0]["label"])], dtype=torch.long)  # [1]

with torch.no_grad():
    outputs_q6 = mc_model(
        input_ids      = input_ids_mc,
        attention_mask = attention_mask_mc,
        labels         = label_row0,
    )

loss_tensor = outputs_q6.loss
num_dims    = loss_tensor.dim()

print(f"\nQ6 — Loss tensor       : {loss_tensor}")
print(f"Q6 — Loss tensor shape : {loss_tensor.shape}")
print(f"Q6 Answer — Number of dimensions: {num_dims}")
# Scalar loss → 0 dimensions (torch.Size([]))

In [ ]:
# Fresh model for LoRA
lora_base = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS,
)

lora_model = get_peft_model(lora_base, lora_config)

trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
all_params       = sum(p.numel() for p in lora_model.parameters())

print(f"\nQ7 — All parameters      : {all_params:,}")
print(f"Q7 — Trainable parameters: {trainable_params:,}")
print(f"Q7 — Trainable %         : {100 * trainable_params / all_params:.4f}%")
print(f"Q7 Answer — Trainable parameters: {trainable_params}")

lora_model.print_trainable_parameters()

In [ ]:
def tokenize_row(row):
    """Tokenize one row into [5, 128] input_ids and attention_mask."""
    opts = [str(row["prompt"]) + " [SEP] " + str(row[c]) for c in CHOICES]
    enc  = tokenizer(
        opts,
        padding       = "max_length",
        truncation    = True,
        max_length    = MAX_LEN,
    )
    return {
        "input_ids"      : enc["input_ids"],        # [5, 128]
        "attention_mask" : enc["attention_mask"],    # [5, 128]
        "labels"         : LABEL2IDX[row["answer"]],
    }

# Build dataset from first 100 rows
rows_100 = [tokenize_row(train.iloc[i]) for i in range(100)]
hf_dataset = Dataset.from_list(rows_100)

# Inspect first item
first_item      = hf_dataset[0]
input_ids_shape = len(first_item["input_ids"]), len(first_item["input_ids"][0])
num_choices     = input_ids_shape[0]

print(f"\nQ8 — First item input_ids shape : {input_ids_shape}")
print(f"Q8 — Dataset size               : {len(hf_dataset)}")
print(f"Q8 Answer — Tokenized choices in input_ids: {num_choices}")

In [ ]:
MAX_LEN_TINY = 64

def tokenize_row_tiny(row):
    opts = [str(row["prompt"]) + " [SEP] " + str(row[c]) for c in CHOICES]
    enc  = tokenizer(
        opts,
        padding       = "max_length",
        truncation    = True,
        max_length    = MAX_LEN_TINY,
    )
    return {
        "input_ids"      : enc["input_ids"],
        "attention_mask" : enc["attention_mask"],
        "labels"         : LABEL2IDX[row["answer"]],
    }

rows_32    = [tokenize_row_tiny(train.iloc[i]) for i in range(32)]
tiny_ds    = Dataset.from_list(rows_32)

# Set tensor format for Trainer
tiny_ds = tiny_ds.with_format("torch")

# Fresh LoRA model for training
lora_base_q9 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
lora_model_q9 = get_peft_model(lora_base_q9, LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS,
))

training_args = TrainingArguments(
    output_dir                  = "./lora_tiny_output",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps                   = 4,
    logging_steps               = 1,
    save_steps                  = 999,       # don't save mid-run
    report_to                   = "none",    # disable wandb for this tiny run
    fp16                        = torch.cuda.is_available(),
)
trainer = Trainer(
    model         = lora_model_q9,
    args          = training_args,
    train_dataset = tiny_ds,
)

train_result  = trainer.train()
global_step   = train_result.global_step

print(f"\nQ9 Answer — Final global_step: {global_step}")

In [ ]:
lora_model_q9.eval()

opts_row0 = [str(row0["prompt"]) + " [SEP] " + str(row0[c]) for c in CHOICES]
enc_q10   = tokenizer(
    opts_row0,
    padding       = "max_length",
    truncation    = True,
    max_length    = MAX_LEN_TINY,
    return_tensors= "pt",
)

input_ids_q10      = enc_q10["input_ids"].unsqueeze(0)       # [1, 5, 64]
attention_mask_q10 = enc_q10["attention_mask"].unsqueeze(0)  # [1, 5, 64]

device = next(lora_model_q9.parameters()).device
input_ids_q10      = input_ids_q10.to(device)
attention_mask_q10 = attention_mask_q10.to(device)

with torch.no_grad():
    outputs_q10 = lora_model_q9(
        input_ids      = input_ids_q10,
        attention_mask = attention_mask_q10,
    )

logits_q10 = outputs_q10.logits            # [1, 5]
probs_q10  = torch.softmax(logits_q10, dim=-1).squeeze(0)  # [5]

print(f"\nQ10 — Logits : {logits_q10}")
print(f"Q10 — Probabilities per option:")
for i, c in enumerate(CHOICES):
    print(f"  Option {c}: {probs_q10[i].item():.4f}")

prob_E = probs_q10[4].item()   # index 4 = E
print(f"\nQ10 Answer — Probability of Option E: {round(prob_E, 4)}")